# Time Series Diagnostics and Volatility Extensions

Module: Financial Time Series

## Lesson summary

Time series modeling is a diagnostic workflow, not a single model choice. This lesson builds a technical checklist for stationarity, ARIMA modeling, residual diagnostics, conditional heteroskedasticity, asymmetric volatility, and model limitations {cite}`box2015time,hamilton1994time,tsay2010analysis`.

## Learning objectives

By the end of this lesson, students should be able to:

- define weak stationarity and explain why raw prices often violate it;
- use log returns as a stationarity-oriented transformation;
- interpret ADF, ACF, and PACF diagnostics;
- fit ARIMA models with model-selection criteria;
- validate residuals with Ljung-Box and Jarque-Bera tests;
- explain GARCH persistence and asymmetric volatility extensions;
- connect volatility forecasts with dynamic VaR.

## Setup

Use the deterministic series and diagnostic helpers in the lesson. Keep the same random seed when comparing diagnostics so interpretation changes come from assumptions, not data regeneration.

## Stationarity

A process $\{y_t\}$ is weakly stationary when:

- $E[y_t] = \mu$ is constant;
- $Var(y_t) = \sigma^2 < \infty$ is constant;
- $Cov(y_t, y_{t-h}) = \gamma_h$ depends only on lag $h$.

Financial prices usually fail this condition. A random walk {cite}`hamilton1994time`,

$$
y_t = y_{t-1} + \epsilon_t,
$$

has variance that grows with time. Returns are usually a better modeling object:

$$
r_t = \ln(P_t) - \ln(P_{t-1}).
$$

In [ ]:
import numpy as np
import pandas as pd

from src.time_series_diagnostics import adf_report, jarque_bera_report, ljung_box_report

## ADF diagnostic

The Augmented Dickey-Fuller test evaluates the null hypothesis that a unit root is present. A low p-value provides evidence against the unit-root null {cite}`dickey1979distribution`.

In [ ]:
rng = np.random.default_rng(42)
white_noise = pd.Series(rng.normal(0, 1, 500), name="white_noise")
random_walk = white_noise.cumsum()

pd.DataFrame(
    {
        "white_noise": adf_report(white_noise),
        "random_walk": adf_report(random_walk),
    }
)

## ACF and PACF

The autocorrelation function is:

$$
\rho_h = \frac{\gamma_h}{\gamma_0}.
$$

ACF measures total linear dependence at lag $h$. PACF isolates the direct contribution of a lag after accounting for intermediate lags. In practice:

- AR processes often show PACF cutoffs;
- MA processes often show ACF cutoffs;
- ARMA processes usually show gradual decay in both.

## ARIMA model workflow

ARIMA$(p,d,q)$ combines autoregressive terms, differencing, and moving-average innovations {cite}`box2015time`:

$$
y_t = c + \sum_{i=1}^p \phi_i y_{t-i} + \epsilon_t + \sum_{j=1}^q \theta_j \epsilon_{t-j}.
$$

The modeling workflow is:

1. decide whether the level or transformed series is the modeling target;
2. test stationarity and choose differencing order $d$;
3. use ACF/PACF to propose candidate $p$ and $q$;
4. estimate candidate models;
5. compare AIC, BIC, and forecast error;
6. run residual diagnostics;
7. only then interpret the model.

Modern `statsmodels` ARIMA estimation uses state-space methods and likelihood optimization internally {cite}`statsmodels2010`. Students do not need to implement the Kalman filter in this course, but they should understand that moving-average errors are latent and cannot be estimated with ordinary least squares.

## Residual diagnostics

A fitted time series model is not finished until residuals are checked.

### Ljung-Box

The Ljung-Box statistic tests whether residual autocorrelations are jointly zero {cite}`ljung1978measure`:

$$
Q = n(n+2)\sum_{j=1}^{k}\frac{\hat{\rho}_j^2}{n-j}.
$$

When testing ARIMA residuals, account for estimated AR and MA parameters with `model_df=p+q`.

In [ ]:
ljung_box_report(white_noise, lags=[5, 10], model_df=0)

### Jarque-Bera

Jarque-Bera compares residual skewness and kurtosis against Gaussian values. Financial returns often reject normality because of skewness and excess kurtosis {cite}`jarque1980efficient,tsay2010analysis`.

In [ ]:
jarque_bera_report(white_noise)

## Conditional heteroskedasticity

Financial returns often show volatility clustering. ARCH and GARCH models describe time-varying conditional variance {cite}`engle1982autoregressive,bollerslev1986generalized`.

Before fitting a full conditional-variance model, it helps to separate simpler volatility estimators from model-based forecasts.

![Comparison of historical, moving-average, and exponentially weighted volatility estimators](../../img/generated/ts-sample-volatility-estimators.png)

For GARCH$(1,1)$:

$$
r_t = \mu + \epsilon_t,
$$

$$
\epsilon_t = \sigma_t z_t,
$$

$$
\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2.
$$

The persistence condition is:

$$
\alpha + \beta < 1.
$$

The long-run variance is:

$$
\frac{\omega}{1-\alpha-\beta}.
$$

## Asymmetric volatility

Standard GARCH treats positive and negative shocks of the same magnitude symmetrically. Equity and FX markets often show asymmetric volatility responses {cite}`glosten1993relation,nelson1991conditional`.

GJR-GARCH adds an indicator for negative shocks:

$$
\sigma_t^2 =
\omega
+ \alpha \epsilon_{t-1}^2
+ \gamma I[\epsilon_{t-1}<0]\epsilon_{t-1}^2
+ \beta \sigma_{t-1}^2.
$$

EGARCH models log variance:

$$
\ln(\sigma_t^2)
= \omega
+ \alpha\left(\frac{|\epsilon_{t-1}|}{\sigma_{t-1}} - E\left[\frac{|\epsilon_{t-1}|}{\sigma_{t-1}}\right]\right)
+ \gamma\frac{\epsilon_{t-1}}{\sigma_{t-1}}
+ \beta\ln(\sigma_{t-1}^2).
$$

This keeps variance positive by construction and can model leverage effects.

## Dynamic VaR bridge

Conditional volatility can feed directly into a one-step-ahead VaR estimate:

$$
VaR_{t+1|t}^{p} = \mu_{t+1|t} + \sigma_{t+1|t}Q(p).
$$

With heavy-tailed residuals, $Q(p)$ should come from a Student's t distribution or another appropriate tail model, not automatically from a Gaussian distribution.

## Model limitations

| Risk | Interpretation |
| --- | --- |
| structural breaks | model parameters may change after crises or policy shifts |
| local optima | likelihood optimization can converge to unstable parameter estimates |
| outlier sensitivity | extreme points can dominate ARIMA or GARCH estimation |
| normality failure | Gaussian VaR can underestimate tail risk |
| long-run reversion | GARCH forecasts eventually revert to long-run variance |
| overfitting | low in-sample error may not imply useful forecasts |